# DIAMOND Research Blueprint

This notebook is a downstream research blueprint for ColliderLake. It starts from the existing `muon_db` lakehouse and sketches the layers that turn curated physics tables into a research-grade collider data platform.

DIAMOND stands for:

- Detector and data-quality certification
- Inference-ready physics features
- Analysis marts and event selections
- Multi-object relations and graph structures
- Observability, validation, and lineage
- Novelty and anomaly candidate stores
- Dissemination-ready datasets and reports

In [1]:
from pathlib import Path
import sys

WORKSPACE = Path.cwd()
if WORKSPACE.name == "notebooks":
    WORKSPACE = WORKSPACE.parent
sys.path.insert(0, str(WORKSPACE))

import pandas as pd

from src.access.muon_db import connect_with_tables
from src.research.registry import research_layer_rows

connection, tables = connect_with_tables(WORKSPACE / "data" / "muon_db")
pd.DataFrame(research_layer_rows())

,order,name,purpose,planned_tables
0,1,certified_physics,"Apply luminosity certification, trigger certif...","certified_event, certified_lumi, certified_tri..."
1,2,physics_marts,Create named analysis selections for reusable ...,"mart_single_muon_baseline, mart_z_to_mumu, mar..."
2,3,object_relations,Represent event topology through objects and p...,"event_object, object_pair, muon_jet_relation, ..."
3,4,research_features,"Store deterministic event, object, resonance, ...","feature_event_kinematics, feature_muon_quality..."
4,5,graph_ready,"Materialize graph nodes, edges, globals, and g...","graph_node, graph_edge, graph_global, graph_ev..."
5,6,anomaly_candidates,Capture deterministic and later model-driven u...,"candidate_high_st, candidate_high_met, candida..."
6,7,validation_observability,"Track row counts, cutflows, schema versions, f...","validation_row_counts, validation_selection_fl..."
7,8,publication,"Freeze dataset cards, cutflows, plots, candida...","dataset_cards, cutflow_reports, plot_manifest,..."


## Current Lakehouse State

The current implementation gives us a real base to build on: bronze raw projections, silver cleaned physics objects, and gold curated tables.

In [2]:
rows = []
for table in tables:
    rows.append({
        "layer": table.layer,
        "table": table.name,
        "rows": connection.execute(f"SELECT count(*) FROM {table.name}").fetchone()[0],
    })
pd.DataFrame(rows)

,layer,table,rows
0,bronze,bronze_event,11704
1,bronze,bronze_muon,11704
2,bronze,bronze_jet,11704
3,bronze,bronze_met,11704
4,bronze,bronze_trigger,11704
5,silver,silver_event,6879
6,silver,silver_muon,6178
7,silver,silver_jet,22462
8,silver,silver_met,6879
9,silver,silver_trigger,6879


## Research Questions To Explore

Start with physics-grounded questions before adding advanced models:

1. What is the event yield after each quality and trigger cut?
2. Does the dimuon table reproduce the expected Z resonance structure?
3. Which high-ST events are driven by jet activity, MET, or muon pT?
4. What event regions are sparse in the current dataset?
5. Which object-pair topologies dominate the selected events?
6. Which candidate events remain interesting after certification and validation?

## Candidate Exploration: High ST Events

This is a deterministic anomaly-candidate view. It is not ML; it is a transparent rule-based starting point.

In [3]:
connection.execute("""
SELECT
    event_id,
    n_muons,
    n_jets,
    leading_muon_pt,
    leading_jet_pt,
    MET_pt,
    HT,
    ST
FROM event_summary
ORDER BY ST DESC
LIMIT 25
""").fetchdf()

,event_id,n_muons,n_jets,leading_muon_pt,leading_jet_pt,MET_pt,HT,ST
0,97c2477a4a45867866061e9934c3d3748cf021f0783af2...,1,4,53.346157,439.2500,108.637489,892.875000,1054.858646
1,1e5739c50abd4533fe251a1defc3c29def9b82919475f9...,1,6,176.294907,358.5000,149.836288,706.984375,1033.115570
2,edf191d6dc88c0d114a7c86f02e657b6427e5a905ff58d...,1,8,209.752121,266.0000,89.428726,650.781250,949.962097
3,4724e1316e637e8d2cfcb98d3ae065c077292d0c942835...,1,4,154.801392,313.0000,198.342392,586.906250,940.050034
4,5739de3cbb848232dd5a854e1f47010ac5dfce378b8bdd...,1,5,143.056122,203.6250,155.208939,636.156250,934.421310
5,2ed971b52acba6769140b84911b1e3cef7e29f800e112f...,1,4,235.976105,263.0000,88.297546,594.757812,919.031464
6,200da533544afa1c697f436c1d826d003f81bcd9564365...,1,5,160.885361,218.6250,181.436630,534.734375,877.056366
7,8af97e1b48d4aba02c14159220669e56ece48e39f5686f...,1,5,74.087029,282.2500,54.826473,733.343750,862.257252
8,9b84d1e21394aeba5ad0a4591e0f201d7353ce91be7aa0...,2,3,201.480942,240.8750,14.825298,461.015625,858.511974
9,18e3a38c8b196bf67751a5780a9481130150b77777bdc8...,1,7,212.384232,232.6250,9.063293,625.406250,846.853775


## Candidate Exploration: Z-Like Dimuon Pairs

The `dimuon` table contains opposite-sign cleaned muon pairs. This query finds pairs closest to the nominal Z mass.

In [4]:
connection.execute("""
SELECT
    event_id,
    muon1_pt,
    muon2_pt,
    invariant_mass,
    delta_r,
    abs(invariant_mass - 91.1876) AS z_distance
FROM dimuon
WHERE invariant_mass BETWEEN 70 AND 110
ORDER BY z_distance
LIMIT 25
""").fetchdf()

,event_id,muon1_pt,muon2_pt,invariant_mass,delta_r,z_distance
0,0a2fd18bce4667b185ffa88e96ddae68bda77b5ccb0198...,54.946426,38.150558,91.186551,2.905274,0.001049
1,b512badbab40733e17602c30e3d6ee70cbe335c5b91725...,62.605728,35.888893,91.189083,2.392698,0.001483
2,4047391f8450ded81ec158daeaaca2c7a6afef8c178043...,57.550613,33.620441,91.201426,2.985945,0.013826
3,9bc81c8ac86eb138c06a2c091e94f764e07857bcffb667...,47.403790,44.486706,91.203087,2.888489,0.015487
4,160d40414a9e0cb9dcf3d2efd97ff6520a754379ed6234...,44.665405,36.322296,91.166593,3.279301,0.021007
5,e1b7c6c1453c3ef558fcb0784b3f17b20f77ead9e88080...,52.998783,24.463240,91.162824,3.440562,0.024776
6,2ece69bd80be6c48a4a418e5a1f5649b3bd12bd71259cb...,52.966099,36.648518,91.159624,2.889091,0.027976
7,e9fbcf73f56bb605849b5a908e883861615422cc4d43fd...,38.025387,32.859013,91.155919,3.212373,0.031681
8,ce858b4c46d60d03b5e5bf046f5af9230fee43521b0dd8...,51.996372,38.427830,91.149106,3.100649,0.038494
9,a144f3f8e786ac7671c1c5391455c8166d30334e1c6cbc...,46.922375,44.126877,91.146452,2.950293,0.041148


## Sparse Region Sketch

Sparse regions are useful for deterministic anomaly-candidate discovery. This is a first-pass event-region binning using muon count, jet count, MET, and ST.

In [5]:
connection.execute("""
SELECT
    n_muons,
    n_jets,
    floor(MET_pt / 25) * 25 AS met_bin_low,
    floor(ST / 100) * 100 AS st_bin_low,
    count(*) AS events
FROM event_summary
GROUP BY n_muons, n_jets, met_bin_low, st_bin_low
ORDER BY events ASC, st_bin_low DESC
LIMIT 50
""").fetchdf()

,n_muons,n_jets,met_bin_low,st_bin_low,events
0,1,6,125.0,1000.0,1
1,1,4,100.0,1000.0,1
2,1,4,75.0,900.0,1
3,1,4,175.0,900.0,1
4,1,5,150.0,900.0,1
5,1,8,75.0,900.0,1
6,1,5,175.0,800.0,1
7,2,3,0.0,800.0,1
8,1,5,50.0,800.0,1
9,1,7,0.0,800.0,1


## Next Implementation Target

The strongest next build is the validation and cutflow layer. It should produce explicit row counts for every major selection step before adding graph or anomaly layers.